# Multi-output BNN + Multi-objective Bayesian Optimization

This notebook models three process outputs from the same decision variables:

- quality — maximize,
- energy consumption — minimize,
- cycle time — minimize.

There is generally no single optimum; the target is a **Pareto set**.

```text
process experiments
      ↓
shared multi-output BNN
      ↓
joint posterior samples for 3 outputs
      ↓
quality ↑, -energy ↑, -cycle ↑
      ↓
Pareto set + hypervolume
      ↓
BoTorch qLogEHVI
      ↓
next experiment
```


In [ ]:
from typing import Optional
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, torch.nn as nn, pyro
from torch import Tensor
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.nn import PyroModule, PyroSample
from botorch.models.ensemble import EnsembleModel
from botorch.acquisition.multi_objective.logei import qLogExpectedHypervolumeImprovement
from botorch.optim import optimize_acqf
from botorch.utils.multi_objective.box_decompositions.non_dominated import FastNondominatedPartitioning
from botorch.utils.multi_objective.pareto import is_non_dominated
SEED=42
np.random.seed(SEED); torch.manual_seed(SEED); pyro.set_rng_seed(SEED); torch.set_default_dtype(torch.float64)


## 1. Synthetic multi-output process


In [ ]:
def physical_process(X):
    x1,x2=X[...,0],X[...,1]
    quality=72+20*torch.exp(-18*((x1-.68)**2+(x2-.34)**2))+7*torch.exp(-30*((x1-.30)**2+(x2-.76)**2))+2.5*torch.sin(6*x1)*torch.cos(5*x2)
    energy=38+22*x1+14*x2+8*(x1-x2)**2-5*torch.exp(-20*((x1-.25)**2+(x2-.25)**2))
    cycle=54-18*x1-10*x2+13*(x1-.60)**2+8*(x2-.55)**2+2*torch.sin(5*x2)
    return torch.stack([quality,energy,cycle],-1)
def observe(X):
    sd=torch.tensor([.8,.7,.6],dtype=X.dtype); return physical_process(X)+torch.randn_like(physical_process(X))*sd
def objectives(Y): return torch.stack([Y[...,0],-Y[...,1],-Y[...,2]],-1)
bounds=torch.tensor([[0.,0.],[1.,1.]])
sobol=torch.quasirandom.SobolEngine(2,scramble=True,seed=SEED); train_X=sobol.draw(24); Y_phys=observe(train_X); Z=objectives(Y_phys)
print('Initial nondominated points:',int(is_non_dominated(Z).sum()))


## 2. Shared multi-output BNN

The three outputs share Bayesian hidden features. Residual noise is output-specific but diagonal, so this is not a full multivariate residual-covariance model.


In [ ]:
Zmean=Z.mean(0); Zstd=Z.std(0).clamp_min(1e-6); Y=(Z-Zmean)/Zstd
class MultiBNN(PyroModule):
    def __init__(self,d=2,h=24,m=3):
        super().__init__(); self.m=m; self.h=PyroModule[nn.Linear](d,h); self.o=PyroModule[nn.Linear](h,m)
        self.h.weight=PyroSample(dist.Normal(0,.9).expand([h,d]).to_event(2)); self.h.bias=PyroSample(dist.Normal(0,.9).expand([h]).to_event(1))
        self.o.weight=PyroSample(dist.Normal(0,.8).expand([m,h]).to_event(2)); self.o.bias=PyroSample(dist.Normal(0,.8).expand([m]).to_event(1))
    def forward(self,X,Y=None):
        mu=self.o(torch.tanh(self.h(X))); pyro.deterministic('mu',mu); sigma=pyro.sample('sigma',dist.LogNormal(-2.2,.35).expand([self.m]).to_event(1))
        with pyro.plate('data',X.shape[0]): pyro.sample('obs',dist.Normal(mu,sigma).to_event(1),obs=Y)
        return mu
pyro.clear_param_store(); pm=MultiBNN(); guide=AutoDiagonalNormal(pm); svi=SVI(pm,guide,pyro.optim.Adam({'lr':.015}),loss=Trace_ELBO())
losses=[svi.step(train_X,Y)/len(train_X) for _ in range(2200)]
plt.figure(figsize=(8,3)); plt.plot(losses); plt.xlabel('SVI step'); plt.ylabel('ELBO / observation'); plt.show()


## 3. BoTorch ensemble wrapper


In [ ]:
class MultiEnsemble(EnsembleModel):
    _num_outputs=3
    def __init__(self,model,guide,S=128): super().__init__(); self.model=model; self.guide=guide; self.S=S
    def forward(self,X):
        batch=X.shape[:-2]; q=X.shape[-2]; d=X.shape[-1]; flat=X.reshape(-1,d)
        mu=Predictive(self.model,guide=self.guide,num_samples=self.S,return_sites=('mu',))(flat)['mu']
        z=mu*Zstd+Zmean; z=z.reshape(self.S,*batch,q,3).movedim(0,len(batch)); return z
model=MultiEnsemble(pm,guide,128)


## 4. Current Pareto set and reference point


In [ ]:
mask=is_non_dominated(Z); pareto=Z[mask]; ref=Z.min(0).values-0.10*(Z.max(0).values-Z.min(0).values).clamp_min(1.0)
print('Reference point:',ref.numpy().round(2)); print('Pareto points:',len(pareto))


## 5. qLogExpectedHypervolumeImprovement

qLogEHVI selects the candidate with high expected improvement in dominated hypervolume, balancing exploration and exploitation across all objectives.


In [ ]:
partitioning=FastNondominatedPartitioning(ref_point=ref,Y=Z)
acq=qLogExpectedHypervolumeImprovement(model=model,ref_point=ref.tolist(),partitioning=partitioning)
candidate,value=optimize_acqf(acq_function=acq,bounds=bounds,q=1,num_restarts=10,raw_samples=128)
new_phys=observe(candidate); print('Suggested setting:',candidate.detach().numpy().round(4)); print('New physical outputs [quality, energy, cycle]:',new_phys.detach().numpy().round(3)); print('Acquisition:',float(value))


## 6. Visualize the current quality-energy trade-off


In [ ]:
pmask=is_non_dominated(Z); plt.figure(figsize=(7,5)); plt.scatter(Y_phys[:,1].numpy(),Y_phys[:,0].numpy(),alpha=.65,label='Experiments'); plt.scatter(Y_phys[pmask,1].numpy(),Y_phys[pmask,0].numpy(),marker='x',s=90,label='3-objective Pareto points'); plt.xlabel('Energy (lower is better)'); plt.ylabel('Quality (higher is better)'); plt.legend(); plt.show()


## Takeaway

Multi-output probabilistic modeling and multi-objective decision making are separate concepts. The BNN supplies uncertainty over several responses; Pareto dominance, the reference point, and hypervolume define the OR decision logic.
